# Decile portfolios from CNN predictions

In this section, I construct 10 decile portfolios from the model's predicted probability of an up move (`pred_prob_up`).

The logic is:

1. Use `end_date` as the portfolio formation date.
2. Compute the realized 5-day forward return from `close_now` and `close_future`.
3. On each rebalance date, sort all stocks into 10 deciles based on `pred_prob_up`.
4. Compute the equal-weight return of each decile.
5. Annualize the mean return and Sharpe ratio.

This follows the general setup in Jiang (2023) and my supervisor's paper, where stocks are sorted into decile portfolios using model predictions, the portfolios are equal-weighted, and held for five trading days.

In [1]:
import pandas as pd
import numpy as np

## 1. Load the CSV and check the required columns

The file contains one row per stock-image observation.  
The most important columns here are:

- `end_date`: the date on which the image ends and the portfolio is formed
- `pred_prob_up`: the model's predicted probability that the future return is positive
- `close_now`: the close price at portfolio formation
- `close_future`: the close price 5 trading days later

I first load the data and make sure the required columns are present.

In [2]:
# Change this to your own file name
csv_path = r"test_res_96_color_candlestick_vol_1_ma_1_bb_0_rsi_0\test_predictions.csv"

df = pd.read_csv(csv_path)

required_cols = [
    "ticker", "end_date", "close_now", "close_future", "pred_prob_up"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Parse date
df["end_date"] = pd.to_datetime(df["end_date"])

# Keep only the columns we need here
df = df[["ticker", "end_date", "close_now", "close_future", "pred_prob_up"]].copy()

print(df.head())
print("\nNumber of rows:", len(df))
print("Number of unique end_date values:", df["end_date"].nunique())
print("Number of unique tickers:", df["ticker"].nunique())

   ticker   end_date  close_now  close_future  pred_prob_up
0   10078 2001-02-01    31.1250       25.8750      0.515768
1   10078 2001-02-08    25.8750       27.1875      0.581099
2   10078 2001-02-15    27.1875       20.8125      0.532564
3   10078 2001-02-23    20.8125       19.6250      0.561816
4   10078 2001-03-02    19.6250       17.4375      0.528365

Number of rows: 71262
Number of unique end_date values: 725
Number of unique tickers: 701


## 2. Compute the realized 5-day forward return

For each stock observation, I compute the realized holding-period return as

$$
R_{i,t \to t+5} = \frac{\text{close\_future}_{i,t}}{\text{close\_now}_{i,t}} - 1
$$

This is the return that each decile portfolio earns over the next 5 trading days.

In [3]:
# Basic cleaning
df = df.dropna(subset=["ticker", "end_date", "close_now", "close_future", "pred_prob_up"]).copy()
df = df[(df["close_now"] > 0) & (df["close_future"] > 0)].copy()

# Remove duplicate stock-date observations if any
df = df.sort_values(["end_date", "ticker"]).drop_duplicates(subset=["ticker", "end_date"])

# Realized 5-day forward return
df["forward_return_5d"] = df["close_future"] / df["close_now"] - 1

print(df[["ticker", "end_date", "close_now", "close_future", "pred_prob_up", "forward_return_5d"]].head())

     ticker   end_date  close_now  close_future  pred_prob_up  \
0     10078 2001-02-01    31.1250        25.875      0.515768   
146   10104 2001-02-01    30.0625        27.125      0.530749   
292   10107 2001-02-01    62.3750        62.250      0.488441   
437   10108 2001-02-01    49.5000        52.800      0.509135   
756   10145 2001-02-01    47.7400        48.780      0.510319   

     forward_return_5d  
0            -0.168675  
146          -0.097713  
292          -0.002004  
437           0.066667  
756           0.021785  


## 3A. Choose rebalance dates when the raw windows overlap

In this case, the dataset contains observations on many consecutive trading days, so the
underlying 5-day windows overlap.

However, if I want portfolio returns that do **not** overlap across holding periods, I should
rebalance every 5th trading date. This gives a clean sequence of non-overlapping 5-day
portfolio returns.

In [ ]:
# Sort unique formation dates
unique_dates = np.sort(df["end_date"].unique())

# Keep every 5th trading date so that portfolio holding periods do not overlap
rebalance_dates = unique_dates[::5]

weekly_df = df[df["end_date"].isin(rebalance_dates)].copy()

print("Total unique dates in full sample:", len(unique_dates))
print("Rebalance dates used:", len(rebalance_dates))
print("Rows after keeping every 5th date:", len(weekly_df))
print(weekly_df.head())

Total unique dates in full sample: 442
Rebalance dates used: 89
Rows after keeping every 5th date: 3291
      ticker   end_date  close_now  close_future  pred_prob_up  \
13584      A 2020-01-30      84.38         84.82      0.592107   
8674    ADSK 2020-01-30     198.99        206.00      0.602067   
7451     AEM 2020-01-30      61.17         59.60      0.500075   
3902    AMAT 2020-01-30      60.25         63.19      0.639207   
3041    ANET 2020-01-30     231.74        232.52      0.577938   

       forward_return_5d  
13584           0.005215  
8674            0.035228  
7451           -0.025666  
3902            0.048797  
3041            0.003366  


## 3B. Choose rebalance dates when the data already has no 5-day overlap

In this case, the dataset is already constructed so that observations do not overlap across
the 5-day horizon.

Therefore, I can use **all available `end_date` values** as rebalance dates. No additional
subsampling is needed.

In [4]:
# Sort unique formation dates
unique_dates = np.sort(df["end_date"].unique())

# Use all dates, because the dataset is already non-overlapping
rebalance_dates = unique_dates

weekly_df = df[df["end_date"].isin(rebalance_dates)].copy()

print("Total unique dates in full sample:", len(unique_dates))
print("Rebalance dates used:", len(rebalance_dates))
print("Rows kept:", len(weekly_df))
print(weekly_df.head())

Total unique dates in full sample: 725
Rebalance dates used: 725
Rows kept: 71261
     ticker   end_date  close_now  close_future  pred_prob_up  \
0     10078 2001-02-01    31.1250        25.875      0.515768   
146   10104 2001-02-01    30.0625        27.125      0.530749   
292   10107 2001-02-01    62.3750        62.250      0.488441   
437   10108 2001-02-01    49.5000        52.800      0.509135   
756   10145 2001-02-01    47.7400        48.780      0.510319   

     forward_return_5d  
0            -0.168675  
146          -0.097713  
292          -0.002004  
437           0.066667  
756           0.021785  


In [5]:
print("COLUMNS:")
print(weekly_df.columns.tolist())

print("\nINDEX NAMES:")
print(weekly_df.index.names)

COLUMNS:
['ticker', 'end_date', 'close_now', 'close_future', 'pred_prob_up', 'forward_return_5d']

INDEX NAMES:
[None]


In [6]:
print("Type:", type(weekly_df))
print("\nColumns as repr:")
print([repr(c) for c in weekly_df.columns])

print("\nIndex names:")
print(weekly_df.index.names)

print("\nDoes exact 'end_date' exist in columns?")
print("end_date" in weekly_df.columns)

Type: <class 'pandas.DataFrame'>

Columns as repr:
["'ticker'", "'end_date'", "'close_now'", "'close_future'", "'pred_prob_up'", "'forward_return_5d'"]

Index names:
[None]

Does exact 'end_date' exist in columns?
True


In [7]:
counts_per_date = weekly_df.groupby("end_date").size()
print(counts_per_date)


end_date
2001-02-01    464
2001-02-02      6
2001-02-05      4
2001-02-07      2
2001-02-08    461
             ... 
2003-12-17     35
2003-12-18     38
2003-12-19     32
2003-12-22     44
2003-12-23    336
Length: 725, dtype: int64


## 4. Assign stocks to 10 deciles on each rebalance date

On each `end_date`, I sort stocks by `pred_prob_up`:

- Decile 1 = lowest predicted probability of going up
- Decile 10 = highest predicted probability of going up

I use `pd.qcut()` to split the cross-section into 10 approximately equal-sized groups.

A small practical issue is that some stocks can have identical prediction values.  
To make `qcut()` stable, I first rank the predictions using `rank(method="first")`.

In [8]:
# Make sure end_date is a normal column, not an index
if "end_date" not in weekly_df.columns:
    weekly_df = weekly_df.reset_index()

# Make a safe copy of end_date to use for grouping
weekly_df["end_date_copy"] = weekly_df["end_date"]

# Keep only dates with at least 10 stocks, otherwise 10 deciles are impossible
counts_per_date = weekly_df.groupby("end_date_copy").size()
valid_dates = counts_per_date[counts_per_date >= 10].index

weekly_df = weekly_df[weekly_df["end_date_copy"].isin(valid_dates)].copy()

def assign_deciles_one_date(group):
    group = group.copy()

    # Rank first to avoid problems when many predictions are tied
    ranked = group["pred_prob_up"].rank(method="first")

    # Decile 1 = lowest prediction, Decile 10 = highest prediction
    group["decile"] = pd.qcut(ranked, q=10, labels=False) + 1
    return group

weekly_df = (
    weekly_df
    .groupby("end_date_copy", group_keys=False)
    .apply(assign_deciles_one_date)
    .reset_index(drop=True)
)

print(weekly_df.head(10))
print("\nNumber of rows:", len(weekly_df))

   ticker   end_date  close_now  close_future  pred_prob_up  \
0   10078 2001-02-01   31.12500       25.8750      0.515768   
1   10104 2001-02-01   30.06250       27.1250      0.530749   
2   10107 2001-02-01   62.37500       62.2500      0.488441   
3   10108 2001-02-01   49.50000       52.8000      0.509135   
4   10145 2001-02-01   47.74000       48.7800      0.510319   
5   10147 2001-02-01   77.60000       59.5000      0.501951   
6   10299 2001-02-01   61.23438       56.6875      0.518920   
7   10324 2001-02-01   85.06250       88.0625      0.530367   
8   10401 2001-02-01   24.46000       22.9000      0.531506   
9   10516 2001-02-01   14.74000       15.9600      0.469629   

   forward_return_5d  decile  
0          -0.168675       7  
1          -0.097713       9  
2          -0.002004       3  
3           0.066667       6  
4           0.021785       6  
5          -0.233247       5  
6          -0.074254       7  
7           0.035268       9  
8          -0.063778       

In [9]:
print("COLUMNS:")
print(list(weekly_df.columns))

print("\nINDEX NAME(S):")
print(weekly_df.index.names)

print("\nHEAD:")
print(weekly_df.head())

COLUMNS:
['ticker', 'end_date', 'close_now', 'close_future', 'pred_prob_up', 'forward_return_5d', 'decile']

INDEX NAME(S):
[None]

HEAD:
   ticker   end_date  close_now  close_future  pred_prob_up  \
0   10078 2001-02-01    31.1250        25.875      0.515768   
1   10104 2001-02-01    30.0625        27.125      0.530749   
2   10107 2001-02-01    62.3750        62.250      0.488441   
3   10108 2001-02-01    49.5000        52.800      0.509135   
4   10145 2001-02-01    47.7400        48.780      0.510319   

   forward_return_5d  decile  
0          -0.168675       7  
1          -0.097713       9  
2          -0.002004       3  
3           0.066667       6  
4           0.021785       6  


In [10]:
# Quick diagnostic: number of stocks in each decile on each date
decile_counts = weekly_df.groupby(["end_date", "decile"]).size().unstack()
print(decile_counts.head())

decile      1   2   3   4   5   6   7   8   9   10
end_date                                          
2001-02-01  47  46  46  47  46  46  47  46  46  47
2001-02-08  47  46  46  46  46  46  46  46  46  46
2001-02-15  46  46  46  45  46  46  45  46  46  46
2001-02-23  46  45  45  45  45  45  45  45  45  45
2001-02-27   1   1   1   1   1   1   1   1   1   1


## 5. Compute equal-weight decile returns on each rebalance date

For each formation date and decile, I take the simple average of the realized 5-day forward returns across all stocks in that decile.

This gives me one 5-day portfolio return for each decile on each rebalance date.

In [11]:
decile_returns_by_date = (
    weekly_df
    .groupby(["end_date", "decile"])["forward_return_5d"]
    .mean()
    .reset_index()
    .sort_values(["end_date", "decile"])
)

print(decile_returns_by_date.head(15))

     end_date  decile  forward_return_5d
0  2001-02-01       1          -0.020626
1  2001-02-01       2          -0.023266
2  2001-02-01       3          -0.030221
3  2001-02-01       4          -0.020624
4  2001-02-01       5          -0.051278
5  2001-02-01       6          -0.024225
6  2001-02-01       7          -0.028937
7  2001-02-01       8          -0.022594
8  2001-02-01       9          -0.017273
9  2001-02-01      10          -0.004348
10 2001-02-08       1          -0.010173
11 2001-02-08       2          -0.006847
12 2001-02-08       3          -0.005142
13 2001-02-08       4          -0.008759
14 2001-02-08       5          -0.005357


## 6. Put the decile returns into wide format and construct High-minus-Low

Now I reshape the data so that each column is one decile:

- column 1 = Low
- column 10 = High

Then I create the spread portfolio:

$$H-L = \text{Decile 10} - \text{Decile 1}$$

This is the same long-short spread that is reported in the papers.

In [12]:
decile_matrix = decile_returns_by_date.pivot(
    index="end_date",
    columns="decile",
    values="forward_return_5d"
).sort_index()

# Add High-minus-Low spread
decile_matrix["H-L"] = decile_matrix[10] - decile_matrix[1]

# Optional: rename columns for nicer display
decile_matrix = decile_matrix.rename(columns={1: "Low", 10: "High"})

print(decile_matrix.head())

decile           Low         2         3         4         5         6  \
end_date                                                                 
2001-02-01 -0.020626 -0.023266 -0.030221 -0.020624 -0.051278 -0.024225   
2001-02-08 -0.010173 -0.006847 -0.005142 -0.008759 -0.005357  0.002369   
2001-02-15 -0.040614 -0.031051 -0.041275 -0.055203 -0.060649 -0.056948   
2001-02-23 -0.003196 -0.003466  0.001962 -0.017922 -0.008942 -0.015652   
2001-02-27 -0.048215 -0.032064  0.058824 -0.035838 -0.089475 -0.064224   

decile             7         8         9      High       H-L  
end_date                                                      
2001-02-01 -0.028937 -0.022594 -0.017273 -0.004348  0.016278  
2001-02-08 -0.002668  0.004353  0.027621 -0.005219  0.004954  
2001-02-15 -0.085874 -0.085595 -0.072941 -0.060879 -0.020265  
2001-02-23  0.002167 -0.028082 -0.017675  0.013118  0.016315  
2001-02-27  0.004412  0.053616  0.026015  0.096174  0.144389  


## 7. Compute annualized return and annualized Sharpe ratio

Each row in `decile_matrix` is a **5-trading-day portfolio return**.

So I annualize using:

$$
\text{periods per year} = \frac{252}{5}
$$

Then:

$$
\text{Annualized Return} = \bar{r}_{5d} \times \frac{252}{5}
$$

$$
\text{Annualized Sharpe} = \frac{\bar{r}_{5d}}{\sigma_{5d}} \times \sqrt{\frac{252}{5}}
$$

This is the standard way to annualize fixed-horizon portfolio returns.

In [13]:
periods_per_year = 252 / 5  # 5-trading-day holding period

def annualized_stats(return_series):
    s = pd.Series(return_series).dropna()
    
    mean_5d = s.mean()
    std_5d = s.std(ddof=1)
    
    ann_return = mean_5d * periods_per_year
    
    if std_5d == 0 or np.isnan(std_5d):
        ann_sharpe = np.nan
    else:
        ann_sharpe = (mean_5d / std_5d) * np.sqrt(periods_per_year)
    
    return pd.Series({
        "Mean_5d_Return": mean_5d,
        "Std_5d_Return": std_5d,
        "Annualized_Return": ann_return,
        "Annualized_Sharpe": ann_sharpe,
        "N_periods": len(s)
    })

summary = decile_matrix.apply(annualized_stats, axis=0).T
summary

,Mean_5d_Return,Std_5d_Return,Annualized_Return,Annualized_Sharpe,N_periods
decile,,,,,
Low,-0.002385,0.037631,-0.120223,-0.450017,699.0
2,-0.000945,0.042439,-0.047614,-0.158036,699.0
3,0.002470,0.044250,0.124511,0.396350,699.0
4,0.000951,0.041327,0.047939,0.163399,699.0
5,0.001666,0.039861,0.083977,0.296756,699.0
6,0.003079,0.050133,0.155201,0.436070,699.0
7,0.001278,0.045713,0.064423,0.198512,699.0
8,0.005256,0.046152,0.264911,0.808526,699.0
9,0.005929,0.047004,0.298797,0.895413,699.0


## 8. Make the final table look like the paper

To make the output easier to compare with the paper tables, I keep only the annualized return and annualized Sharpe ratio, and I convert the return to percent.

In [14]:
final_table = summary[["Annualized_Return", "Annualized_Sharpe", "N_periods"]].copy()
final_table["Annualized_Return_pct"] = final_table["Annualized_Return"] * 100

# Put columns in a nicer order
final_table = final_table[["Annualized_Return_pct", "Annualized_Sharpe", "N_periods"]]

# Optional: reorder rows
desired_order = ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"]
final_table = final_table.reindex([x for x in desired_order if x in final_table.index])

print(final_table.round(4))

        Annualized_Return_pct  Annualized_Sharpe  N_periods
decile                                                     
Low                  -12.0223            -0.4500      699.0
2                     -4.7614            -0.1580      699.0
3                     12.4511             0.3964      699.0
4                      4.7939             0.1634      699.0
5                      8.3977             0.2968      699.0
6                     15.5201             0.4361      699.0
7                      6.4423             0.1985      699.0
8                     26.4911             0.8085      699.0
9                     29.8797             0.8954      699.0
High                  40.3688             1.3106      699.0
H-L                   52.3911             1.7190      699.0


## 9. Interpretation of the output

The final table should be read as follows:

- `Low` is the decile with the lowest predicted probability of an up move.
- `High` is the decile with the highest predicted probability of an up move.
- `H-L` is a long-short strategy that buys the highest decile and shorts the lowest decile.
- `Annualized_Return_pct` is the annualized mean return in percent.
- `Annualized_Sharpe` is the annualized Sharpe ratio.

If the model is useful, I should generally see returns and Sharpe ratios improve as I move from `Low` to `High`.

In [15]:
# Average number of stocks in each decile
avg_names_per_decile = (
    weekly_df.groupby(["end_date", "decile"]).size()
    .groupby("decile")
    .mean()
)

print("Average number of stocks per decile:")
print(avg_names_per_decile.round(2))

# Check monotonicity visually
display_cols = [c for c in ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"] if c in decile_matrix.columns]
display(decile_matrix[display_cols].head())

Average number of stocks per decile:
decile
1     10.60
2     10.12
3     10.03
4     10.12
5     10.22
6      9.93
7     10.03
8     10.11
9     10.04
10    10.53
dtype: float64


decile,Low,2,3,4,5,6,7,8,9,High,H-L
end_date,,,,,,,,,,,
2001-02-01,-0.020626,-0.023266,-0.030221,-0.020624,-0.051278,-0.024225,-0.028937,-0.022594,-0.017273,-0.004348,0.016278
2001-02-08,-0.010173,-0.006847,-0.005142,-0.008759,-0.005357,0.002369,-0.002668,0.004353,0.027621,-0.005219,0.004954
2001-02-15,-0.040614,-0.031051,-0.041275,-0.055203,-0.060649,-0.056948,-0.085874,-0.085595,-0.072941,-0.060879,-0.020265
2001-02-23,-0.003196,-0.003466,0.001962,-0.017922,-0.008942,-0.015652,0.002167,-0.028082,-0.017675,0.013118,0.016315
2001-02-27,-0.048215,-0.032064,0.058824,-0.035838,-0.089475,-0.064224,0.004412,0.053616,0.026015,0.096174,0.144389


## Always-long equal-weight benchmark

As a reference benchmark, I also calculate the performance of an **always-long, equal-weight portfolio across all stocks**.

This benchmark does **not** sort stocks into deciles.  
Instead, on each portfolio formation date, it simply buys **all available stocks** and assigns each stock the same weight.

That is why this benchmark produces only **one portfolio return series**, and therefore only **one annualized return** and **one Sharpe ratio**.

### Step 1: Compute the 5-day forward return for each stock

For each stock $i$ on formation date $t$, the realized 5-day forward return is:

$$
r_{i,t} = \frac{\text{close\_future}_{i,t}}{\text{close\_now}_{i,t}} - 1
$$

where:

- $\text{close\_now}_{i,t}$ is the closing price at the portfolio formation date
- $\text{close\_future}_{i,t}$ is the closing price 5 trading days later

### Step 2: Compute the equal-weight benchmark return on each date

On each formation date $t$, the always-long benchmark return is the simple average of all stock returns on that date:

$$
r^{EW}_t = \frac{1}{N_t} \sum_{i=1}^{N_t} r_{i,t}
$$

where $N_t$ is the number of available stocks on date $t$.

So instead of creating 10 decile portfolios, I create only **one** portfolio each period:

- long all stocks
- equal weight each stock
- hold for 5 trading days

This gives a time series of benchmark returns:

$$
r^{EW}_{t_1}, r^{EW}_{t_2}, r^{EW}_{t_3}, \dots
$$

### Step 3: Annualize the mean return

Because each portfolio is held for 5 trading days, the number of holding periods per year is approximately:

$$
\frac{252}{5}
$$

where 252 is the standard number of trading days in a year.

The annualized return is therefore:

$$
\text{Annualized Return} = \bar{r}_{5d} \times \frac{252}{5}
$$

where $\bar{r}_{5d}$ is the average 5-day benchmark return across all periods.

### Step 4: Annualize the Sharpe ratio

The Sharpe ratio measures return relative to volatility.

Let $\sigma_{5d}$ denote the standard deviation of the 5-day benchmark returns.  
Then the annualized Sharpe ratio is:

$$
\text{Annualized Sharpe} = \frac{\bar{r}_{5d}}{\sigma_{5d}} \times \sqrt{\frac{252}{5}}
$$

### Why does this benchmark only give one return and one Sharpe ratio?

The decile analysis gives many values because stocks are split into many portfolios:

- Decile 1
- Decile 2
- ...
- Decile 10
- High-minus-Low

So each decile has its own return series and its own Sharpe ratio.

In contrast, the always-long benchmark does **not** split stocks into groups.  
It simply averages all stocks into one equal-weight portfolio on each date.

Therefore, it produces:

- one portfolio return series
- one annualized return
- one annualized Sharpe ratio

### Interpretation

This benchmark is useful because it shows how well a simple passive strategy performs without using the model.

If the model is useful in a long-only sense, then the **High decile** should ideally outperform this benchmark in terms of:

- annualized return
- Sharpe ratio

If the model is useful as a ranking model, then returns should generally improve from the **Low** decile to the **High** decile, and the **High-minus-Low** spread should be positive and economically meaningful.

In [16]:
import pandas as pd
import numpy as np

# df must contain: end_date, close_now, close_future
benchmark_df = df.copy()

benchmark_df["end_date"] = pd.to_datetime(benchmark_df["end_date"])
benchmark_df = benchmark_df.dropna(subset=["end_date", "close_now", "close_future"])
benchmark_df = benchmark_df[(benchmark_df["close_now"] > 0) & (benchmark_df["close_future"] > 0)].copy()

# 5-day forward stock return
benchmark_df["forward_return_5d"] = benchmark_df["close_future"] / benchmark_df["close_now"] - 1

# Always-long equal-weight portfolio across all stocks on each date
ew_returns = (
    benchmark_df
    .groupby("end_date")["forward_return_5d"]
    .mean()
    .sort_index()
)

# Annualization for 5-trading-day holding periods
periods_per_year = 252 / 5

annualized_return = ew_returns.mean() * periods_per_year
annualized_sharpe = (ew_returns.mean() / ew_returns.std(ddof=1)) * np.sqrt(periods_per_year)

print("Always-long equal-weight benchmark")
print(f"Annualized return: {annualized_return:.4f}  ({annualized_return*100:.2f}%)")
print(f"Annualized Sharpe: {annualized_sharpe:.4f}")
print(f"Number of periods: {len(ew_returns)}")

Always-long equal-weight benchmark
Annualized return: 0.1050  (10.50%)
Annualized Sharpe: 0.4571
Number of periods: 725
